In [1]:
from pathlib import Path
import math, json
import numpy as np
import trimesh
from shapely.geometry import Polygon

# Prefer OpenSCAD if available for robust booleans
# ENGINE = 'scad' if trimesh.interfaces.scad.exists else None
ENGINE = None

# --- Your scale params (same as notebook) ---
XY_SCALE        = 2/100000
Z_EXAGGERATION  = 1.5
Z_SCALE         = XY_SCALE * Z_EXAGGERATION

# Convert real meters -> model millimeters
XY_SCALE_MM_PER_M = XY_SCALE * 1000.0
Z_SCALE_MM_PER_M  = Z_SCALE  * 1000.0

# Global base reference you already compute in your pipeline
min_elev_m_global = 1200.0   # <-- use your real value if you have it handy
bottom_offset_m   = 10.0     # same as BOTTOM_OFFSET_M

def z_mm_from_real_elev(e_real_m: float) -> float:
    base_m = min_elev_m_global - bottom_offset_m
    return (e_real_m - base_m) * Z_SCALE_MM_PER_M


In [2]:
# --- in Cell B ---
def add_contour_grooves(
    mesh: trimesh.Trimesh,
    *,
    baseline_m: float,
    interval_m: float,
    band_thickness_mm: float,
) -> tuple[trimesh.Trimesh, list[float]]:
    z_min_mm, z_max_mm = float(mesh.bounds[0,2]), float(mesh.bounds[1,2])
    if interval_m <= 0:
        return mesh, []

    base_m = min_elev_m_global - bottom_offset_m
    e_lo = (z_min_mm / Z_SCALE_MM_PER_M) + base_m
    e_hi = (z_max_mm / Z_SCALE_MM_PER_M) + base_m

    k = math.ceil((e_lo - baseline_m) / interval_m)
    z_levels, engravers = [], []

    pad = 2.0
    dx = (mesh.bounds[1,0] - mesh.bounds[0,0]) + pad
    dy = (mesh.bounds[1,1] - mesh.bounds[0,1]) + pad
    cx = float(mesh.bounds.mean(axis=0)[0])
    cy = float(mesh.bounds.mean(axis=0)[1])

    while True:
        e_k = baseline_m + k * interval_m
        if e_k > e_hi:
            break
        z_k = z_mm_from_real_elev(e_k)
        z_levels.append(z_k)
        T = trimesh.transformations.translation_matrix([cx, cy, z_k])
        engravers.append(trimesh.creation.box(extents=[dx, dy, band_thickness_mm], transform=T))
        k += 1

    if not engravers:
        return mesh, []

    cutter = trimesh.util.concatenate(engravers)
    # ✅ pass a sequence
    diff = trimesh.boolean.difference([mesh, cutter])
    return diff, z_levels


In [3]:
# Text: try trimesh text; fall back to labeled boxes if freetype isn't available
try:
    from trimesh.creation import text as tm_text
    HAVE_TEXT = True
except Exception:
    HAVE_TEXT = False

def make_text_mesh(s: str, height_mm: float, depth_mm: float) -> trimesh.Trimesh:
    if HAVE_TEXT:
        m = tm_text(text=s, font='Arial', font_size=height_mm, depth=depth_mm)
        if isinstance(m, trimesh.Scene):
            m = trimesh.util.concatenate([g for g in m.dump().geometry.values()])
        return m
    # Fallback: sized box (keeps the boolean flow working)
    w = max(1.0, len(s) * height_mm * 0.6)
    return trimesh.creation.box(extents=[w, height_mm, depth_mm])

# --- in Cell C ---
def engrave_bottom(
    mesh: trimesh.Trimesh,
    *,
    text_lines: list[str],
    text_height_mm: float,
    engrave_depth_mm: float,
    offset_xy_mm: tuple[float, float] = (50.0, 50.0),
    north_arrow_size_mm: float = 20.0,
    north_azimuth_deg: float = 0.0
) -> trimesh.Trimesh:
    z_bottom = float(mesh.bounds[0,2])
    cx, cy = float(mesh.bounds.mean(axis=0)[0]), float(mesh.bounds.mean(axis=0)[1])

    line_gap = text_height_mm * 0.35
    engravers = []
    x0 = cx + offset_xy_mm[0]
    y0 = cy + offset_xy_mm[1]
    z0 = z_bottom + engrave_depth_mm/2.0

    for i, line in enumerate(text_lines):
        t = make_text_mesh(line, height_mm=text_height_mm, depth_mm=engrave_depth_mm)
        tb = t.bounds
        tw = float(tb[1,0] - tb[0,0]); th = float(tb[1,1] - tb[0,1])
        T = trimesh.transformations.translation_matrix([x0 + tw/2.0, y0 - i*(th+line_gap) + th/2.0, z0])
        engravers.append(t.apply_transform(T))

    from shapely.geometry import Polygon as SPoly
    arrow_poly = SPoly([(0,0), (north_arrow_size_mm*0.5, north_arrow_size_mm*1.0), (north_arrow_size_mm, 0)])
    arrow = trimesh.creation.extrude_polygon(arrow_poly, height=engrave_depth_mm)
    Rz = trimesh.transformations.rotation_matrix(math.radians(north_azimuth_deg), [0,0,1])
    arrow.apply_transform(Rz)
    aT = trimesh.transformations.translation_matrix([x0, y0 + text_height_mm*1.8, z0])
    engravers.append(arrow.apply_transform(aT))

    cutter = trimesh.util.concatenate(engravers)
    # ✅ pass a sequence
    carved = trimesh.boolean.difference([mesh, cutter])
    return carved

In [4]:
# ── Load your watertight tile (OBJ in mm units) ──────────────────────────────
in_path  = Path("./tmp/obj_export_hex1.obj")   # pick any of the files you showed
mesh     = trimesh.load(in_path, force='mesh')
print("Loaded:", in_path.name, "is_volume:", mesh.is_volume)
print("Bounds (mm):", mesh.bounds)

# ── Contour parameters (edit as desired) ─────────────────────────────────────
baseline_m        = 0.0       # first stripe starts at 0 m real elevation
interval_m        = 250.0     # every 250 m real elevation
band_thickness_mm = 0.6       # physical groove thickness in the print

mesh_bands, z_levels = add_contour_grooves(
    mesh,
    baseline_m=baseline_m,
    interval_m=interval_m,
    band_thickness_mm=band_thickness_mm,
)
print(f"Added {len(z_levels)} grooves.")

# ── Engraving content (derived from your scales) ─────────────────────────────
mm_per_km = XY_SCALE_MM_PER_M * 1000.0
text_lines = [
    f"XY scale: {mm_per_km:.1f} mm / km",
    f"Z-exag: {Z_EXAGGERATION:.2f}x",
    f"Contours: {interval_m:.0f} m / {band_thickness_mm:.2f} mm",
    "Tile: 01"
]

mesh_final = engrave_bottom(
    mesh_bands,
    text_lines=text_lines,
    text_height_mm=4.0,
    engrave_depth_mm=0.8,
    offset_xy_mm=(50.0, 50.0),   # keeps the center free for mounting later
    north_arrow_size_mm=20.0,
    north_azimuth_deg=0.0        # rotate if you know true north relative to +Y
)

Loaded: obj_export_hex1.obj is_volume: True
Bounds (mm): [[  3.92845392  45.86602783   0.        ]
 [ 52.35046005 101.77893829  20.59483337]]
Added 3 grooves.


In [9]:
# ── Save outputs ─────────────────────────────────────────────────────────────
out_dir = Path("tmp").resolve()
out_dir.mkdir(exist_ok=True)
out_path = out_dir / (in_path.stem + "_postprocessed.stl")
mesh_final.export(out_path)

meta = {
    "source": str(in_path),
    "XY_SCALE": XY_SCALE,
    "Z_EXAGGERATION": Z_EXAGGERATION,
    "Z_SCALE": Z_SCALE,
    "XY_SCALE_MM_PER_M": XY_SCALE_MM_PER_M,
    "Z_SCALE_MM_PER_M": Z_SCALE_MM_PER_M,
    "min_elev_m_global": min_elev_m_global,
    "bottom_offset_m": bottom_offset_m,
    "contour": {
        "baseline_m": baseline_m,
        "interval_m": interval_m,
        "band_thickness_mm": band_thickness_mm
    },
    "engraving": {
        "text_height_mm": 4.0,
        "engrave_depth_mm": 0.8,
        "offset_xy_mm": [50.0, 50.0],
        "north_azimuth_deg": 0.0
    }
}
with open(out_dir / (in_path.stem + "_postprocessed.meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", out_path)

Saved: /TouchTerrain/standalone/tmp/obj_export_hex1_postprocessed.stl


In [7]:
# ── Cell E — k3d previews ───────────────────────────────────────────────────
import numpy as np
import k3d

def show_k3d(*meshes, colors=None, wireframe=False, opacities=None, title=None):
    """
    Quick viewer. Pass one or more trimesh.Trimesh objects.
    """
    plot = k3d.plot(grid_visible=False, axes_helper=0)
    if colors is None:
        colors = [0xCCCCCC, 0x6AA9FF, 0xFF9966, 0x66CC99][:len(meshes)]
    if opacities is None:
        opacities = [1.0] * len(meshes)

    for m, c, a in zip(meshes, colors, opacities):
        v = m.vertices.astype(np.float32)
        f = m.faces.astype(np.uint32)
        plot += k3d.mesh(
            vertices=v,
            indices=f,
            color=c,
            flat_shading=False,
            wireframe=wireframe,
            opacity=a,
        )
    if title:
        plot += k3d.text(text=title, position=[0,0,float(np.max(meshes[0].bounds[:,2])) + 5],
                         color=0x777777, size=12)
    plot.display()
    return plot

# # 1) Original tile
# show_k3d(mesh, title="original tile")

# # 2) After adding contour grooves
# show_k3d(mesh_bands, title="with contour grooves")

# 3) Final (grooves + bottom engraving)
show_k3d(mesh_final, title="final (grooves + engraving)")

# # (optional) Overlay comparison: original as faint ghost + final solid
# show_k3d(mesh, mesh_final, opacities=[0.25, 1.0], colors=[0x888888, 0xFF9966],
#          title="overlay: original (ghost) vs final")


Output()

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper_colors=[16711680, 65280, 255], background_color=16777215, …